# **Fase 4b · Comparação com a escuta**
### *Compara os rótulos do PANNs e do CLAP com a tabela de eventos de referência, conforme `docs/fase4/02_desenho_experimental.md`, e monta a lista de trechos para a escuta dirigida.*

(Este caderno não usa os áudios nem a GPU. Lê apenas os JSON produzidos pelo caderno 4a, a tabela de eventos de referência e, na seção 6, os JSON da transcrição da fase 3. Pode ser executado por qualquer pessoa a partir do repositório)

## **1. Preparação**

*Instala o pacote do PANNs apenas para obter a lista das 527 categorias do AudioSet, na ordem em que o modelo as usa, necessária para delimitar o ramo musical (seção 2). Conecta o Drive e define as pastas. A tabela de referência é a mesma versionada em `docs/fase4/01_eventos_referencia.md`, copiada para a pasta `referencias` do Drive, junto com as transcrições de referência da fase 3. As tabelas produzidas aqui são gravadas em `resultados/fase4/medidas`.*

In [1]:
!pip install -q panns-inference==0.1.1

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os, re, json
import pandas as pd

PASTA_PROJETO = '/content/drive/MyDrive/escuta-cerrada-poc'
PASTA_RESULTADOS = os.path.join(PASTA_PROJETO, 'resultados', 'fase4')
PASTA_FASE3 = os.path.join(PASTA_PROJETO, 'resultados', 'fase3')
PASTA_REFERENCIAS = os.path.join(PASTA_PROJETO, 'referencias')
PASTA_MEDIDAS = os.path.join(PASTA_RESULTADOS, 'medidas')
os.makedirs(PASTA_MEDIDAS, exist_ok=True)

print(len([a for a in os.listdir(PASTA_RESULTADOS) if a.endswith('.json')]), 'arquivos JSON na fase 4')
print('01_eventos_referencia.md' in os.listdir(PASTA_REFERENCIAS), '· tabela de referência encontrada')

Mounted at /content/drive
21 arquivos JSON na fase 4
True · tabela de referência encontrada


### 1.1 Leitura das vinte execuções

Lê os JSON das vinte execuções (o arquivo da repetição do teste de determinismo fica de fora) e confere que o PANNs e o CLAP analisaram cada faixa na mesma grade de janelas, condição para comparar os dois modelos janela a janela.

In [3]:
EXECUCOES = {}
for n in range(1, 6):
    for modelo in ['panns', 'clap']:
        for condicao in ['sem', 'com']:
            with open(os.path.join(PASTA_RESULTADOS, f'faixa{n:02d}_{modelo}_{condicao}.json'), encoding='utf-8') as f:
                EXECUCOES[(n, modelo, condicao)] = json.load(f)['janelas']

for n in range(1, 6):
    grades = [[(j['inicio'], j['fim']) for j in EXECUCOES[(n, m, c)]] for m in ['panns', 'clap'] for c in ['sem', 'com']]
    print(f'Faixa {n:02d} · {len(grades[0])} janelas · mesma grade nas quatro execuções:', all(g == grades[0] for g in grades))

Faixa 01 · 335 janelas · mesma grade nas quatro execuções: True
Faixa 02 · 201 janelas · mesma grade nas quatro execuções: True
Faixa 03 · 362 janelas · mesma grade nas quatro execuções: True
Faixa 04 · 64 janelas · mesma grade nas quatro execuções: True
Faixa 05 · 441 janelas · mesma grade nas quatro execuções: True


**Resultado.** As quatro execuções de cada faixa usam a mesma grade de janelas, condição para a comparação janela a janela.

## **2. Regras fixadas no desenho experimental**

*Transcreve para o código as seções 4, 6 e 9 do desenho experimental. `PANNS_CLASSES` e `CLAP_CLASSES` trazem, para cada classe da tabela de referência, os rótulos do AudioSet e as descrições do CLAP que contam como correspondentes. Classes sem nenhum rótulo correspondente (a pausa e a mudança de intensidade nos dois modelos, os pulos no PANNs) ficam de fora das listas e aparecem nos resultados como "sem correspondência". Os rótulos registrados à parte ficam em listas próprias.*

In [4]:
LIMIAR_APONTADO = 0.3     # PANNs, seção 4 do desenho
LIMIAR_INDICIO = 0.1      # PANNs, seção 4 do desenho
TOP_CLAP = 3              # CLAP, seção 4 do desenho
LIMIAR_SEM_CLASSE = 0.5   # PANNs, critério 2 da escuta dirigida (seção 9)
FOLGA_AGRUPAMENTO = 5     # segundos, decisão de 20/10/2026

# Classes presentes do começo ao fim, fora do critério 1 da escuta dirigida (seção 9)
SEMPRE_PRESENTES = {'ruído de disco': {1, 2, 3, 4, 5}, 'fala': {1, 2, 3}}

# Seção 6: rótulos do AudioSet (PANNs) por classe. A classe "música" é completada adiante.
PANNS_CLASSES = {
    'aplauso': ['Applause', 'Clapping'],
    'grito de plateia': ['Cheering', 'Crowd', 'Shout', 'Yell', 'Hubbub, speech noise, speech babble'],
    'tosse': ['Cough', 'Throat clearing'],
    'fala': ['Speech', 'Male speech, man speaking', 'Narration, monologue', 'Conversation'],
    'voz alterada': ['Distortion'],
    'ruído de disco': ['Hiss', 'Static', 'Crackle', 'Noise', 'White noise', 'Pink noise'],
    'estalos regulares': ['Clicking', 'Tick', 'Tick-tock', 'Crackle'],
    'papel': ['Crumpling, crinkling', 'Rustle', 'Tearing'],
    'surdo': ['Bass drum', 'Drum', 'Timpani'],
    'tamborim': ['Drum', 'Snare drum', 'Percussion'],
    'vento': ['Wind', 'Wind noise (microphone)'],
    'canto': ['Singing', 'Female singing', 'Choir', 'Chant', 'Vocal music'],
    'sino': ['Bell', 'Church bell', 'Chime', 'Tubular bells'],
    'passos': ['Walk, footsteps'],
    'zumbido': ['Hum', 'Mains hum', 'Buzz'],
}

# Descrições do CLAP por classe
CLAP_CLASSES = {
    'aplauso': ['applause'],
    'grito de plateia': ['a crowd cheering and shouting'],
    'tosse': ['a man coughing'],
    'fala': ['a man giving a speech'],
    'voz alterada': ['a distorted voice'],
    'ruído de disco': ['the surface noise of an old record'],
    'estalos regulares': ['clicks of a scratched record'],
    'pulos': ['a skipping record'],
    'papel': ['a page of paper being turned'],
    'surdo': ['surdo', 'a surdo, a large Brazilian bass drum', 'a bass drum'],
    'tamborim': ['tamborim', 'a tamborim, a small Brazilian frame drum played with a stick', 'a snare drum'],
    'música': ['music'],
    'vento': ['wind'],
    'canto': ['women singing'],
    'sino': ['a church bell'],
    'passos': ['footsteps'],
    'zumbido': ['electrical hum'],
}

# Rótulos registrados à parte (seção 6)
A_PARTE_PANNS = {
    'Tambourine': 'tamborim (homônimo)', 'Silence': 'pausa', 'Door': 'passos (alternativa)',
    'Knock': 'passos (alternativa)', 'Ocean': 'surpresa da fase 1', 'Waves, surf': 'surpresa da fase 1',
}
A_PARTE_CLAP = {
    'a tambourine': 'tamborim (homônimo)', 'silence': 'pausa',
    'a door knocking': 'passos (alternativa)', 'ocean waves': 'surpresa da fase 1',
}

### 2.1 O ramo musical do AudioSet

A seção 6 do desenho faz corresponder à classe "música" os rótulos de índice 137 a 282 do AudioSet (instrumentos, gêneros e climas musicais), exceto os de percussão já atribuídos ao surdo e ao tamborim. A função abaixo aplica essa regra e retira também os rótulos atribuídos a outras classes (canto, sino) e os registrados à parte (Tambourine). A célula mostra o que ficou de fora do ramo, para conferência.

In [5]:
def rotulos_musica(rotulos_audioset):
    """Rótulos do ramo musical do AudioSet (índices 137 a 282), exceto os atribuídos a outras
    classes ou registrados à parte."""
    outros = {r for lista in PANNS_CLASSES.values() for r in lista} | set(A_PARTE_PANNS)
    return [r for r in rotulos_audioset[137:283] if r not in outros]


def mapa_inverso(classes):
    """{rótulo: {classes}} a partir de {classe: [rótulos]}."""
    inverso = {}
    for classe, rotulos in classes.items():
        for r in rotulos:
            inverso.setdefault(r, set()).add(classe)
    return inverso


from panns_inference.config import labels as ROTULOS_AUDIOSET

PANNS_CLASSES['música'] = rotulos_musica(ROTULOS_AUDIOSET)
INVERSO_PANNS = mapa_inverso(PANNS_CLASSES)
INVERSO_CLAP = mapa_inverso(CLAP_CLASSES)

print(len(PANNS_CLASSES['música']), 'rótulos contam como música, de', ROTULOS_AUDIOSET[137], 'a', ROTULOS_AUDIOSET[282])
print('Retirados do ramo musical:', [r for r in ROTULOS_AUDIOSET[137:283] if r not in PANNS_CLASSES['música']])

135 rótulos contam como música, de Music a Scary music
Retirados do ramo musical: ['Percussion', 'Drum', 'Snare drum', 'Bass drum', 'Timpani', 'Tambourine', 'Tubular bells', 'Bell', 'Church bell', 'Chime', 'Vocal music']


**Resultado.** 135 rótulos do ramo musical contam como música. Os 11 retirados correspondem aos rótulos de percussão atribuídos ao surdo e ao tamborim, aos de sino, a "Vocal music", atribuído ao canto, e a "Tambourine", registrado à parte, conforme a leitura do desenho registrada no diário da fase (20/10).

## **3. Tabela de referência**

*Lê as tabelas de `01_eventos_referencia.md` linha a linha. Os tempos passam a segundos, e as classes compostas ("aplauso e grito de plateia", "fala, música e vento") são desdobradas em componentes, cada um avaliado separadamente, conforme a seção 4 do desenho. Os fenômenos sem localização precisa não têm tempo e não entram na comparação automática.*

In [6]:
def segundos(t):
    """'4:17,15' -> 257.15"""
    m, s = t.strip().split(':')
    return int(m) * 60 + float(s.replace(',', '.'))


def mmss(s):
    """257.15 -> '4:17,2'"""
    return f'{int(s // 60)}:{s % 60:04.1f}'.replace('.', ',')


def ler_referencia(texto):
    """Lê as tabelas de eventos do arquivo markdown. Devolve uma lista de dicionários."""
    eventos = []
    for linha in texto.splitlines():
        m = re.match(r'^\|\s*(\d+)\.(\d+)\s*\|', linha)
        if not m:
            continue
        cel = [c.strip() for c in linha.strip().strip('|').split('|')]
        eventos.append({
            'id': cel[0], 'faixa': int(m.group(1)), 'inicio': segundos(cel[1]), 'fim': segundos(cel[2]),
            'classe': cel[3], 'descricao': cel[4], 'fonte': cel[5],
            'componentes': [c.strip() for c in re.split(r',| e ', cel[3]) if c.strip()],
        })
    return eventos


with open(os.path.join(PASTA_REFERENCIAS, '01_eventos_referencia.md'), encoding='utf-8') as f:
    EVENTOS = ler_referencia(f.read())

ref = pd.DataFrame(EVENTOS)
print(len(EVENTOS), 'eventos ·', ref.groupby('faixa').size().to_dict())
print(sum(len(e['componentes']) for e in EVENTOS), 'componentes depois do desdobramento')
print(ref.explode('componentes')['componentes'].value_counts().to_string())

44 eventos · {1: 13, 2: 5, 3: 5, 4: 7, 5: 14}
53 componentes depois do desdobramento
componentes
aplauso                   8
ruído de disco            8
fala                      5
vento                     5
música                    5
grito de plateia          4
papel                     3
pulos                     2
pausa                     2
tamborim                  2
voz alterada              1
surdo                     1
estalos regulares         1
tosse                     1
canto                     1
sino                      1
passos                    1
mudança de intensidade    1
zumbido                   1


**Resultado.** A tabela tem 44 eventos, que somam 53 componentes depois do desdobramento das classes compostas. As classes mais frequentes são o aplauso e o ruído de disco (oito componentes cada), e a pausa e a mudança de intensidade, sem correspondência nos modelos, aparecem em três componentes.

## **4. Eventos nomeados**

*Para cada componente de cada evento, reúne as janelas que se sobrepõem a ele e verifica se algum rótulo correspondente à sua classe foi apontado. No PANNs, o evento é nomeado quando algum rótulo da classe chega a 0,3 em alguma dessas janelas, e fica como indício quando o máximo fica entre 0,1 e 0,3. No CLAP, o evento é nomeado quando alguma descrição da classe fica entre as três primeiras, e a tabela registra a melhor posição alcançada, mesmo quando não nomeia. Em todos os casos registra-se o rótulo, o valor e o início da janela em que ele ocorreu.*

In [7]:
def sobrepoe(janela, inicio, fim):
    """A janela e o intervalo [inicio, fim] têm algum tempo em comum."""
    return janela['inicio'] < fim and janela['fim'] > inicio


def posicoes(janela):
    """CLAP: posição (1 = mais semelhante) de cada descrição na janela."""
    ordem = sorted(janela['semelhancas'], key=janela['semelhancas'].get, reverse=True)
    return {termo: k for k, termo in enumerate(ordem, start=1)}


def primeiro(janela):
    """CLAP: descrição mais semelhante da janela."""
    return max(janela['semelhancas'], key=janela['semelhancas'].get)


def nomeacao_panns(classe, janelas, classes_panns):
    """Situação de uma classe num conjunto de janelas (as que se sobrepõem a um evento).

    Devolve (situação, rótulo, probabilidade, início da janela). A situação é 'nomeado'
    (algum rótulo da classe com pelo menos 0,3), 'indício' (máximo entre 0,1 e 0,3),
    'não nomeado' ou 'sem correspondência' (classe sem rótulo no AudioSet).
    """
    rotulos = classes_panns.get(classe)
    if not rotulos:
        return 'sem correspondência', None, None, None
    melhor = (None, 0.0, None)
    for j in janelas:
        for r in rotulos:
            p = j['rotulos'].get(r, 0.0)
            if p > melhor[1]:
                melhor = (r, p, j['inicio'])
    if melhor[1] >= LIMIAR_APONTADO:
        situacao = 'nomeado'
    elif melhor[1] >= LIMIAR_INDICIO:
        situacao = 'indício'
    else:
        situacao = 'não nomeado'
    return situacao, melhor[0], round(melhor[1], 4) if melhor[0] else None, melhor[2]


def nomeacao_clap(classe, janelas, classes_clap, k=TOP_CLAP):
    """Como nomeacao_panns, para o CLAP. O valor é a melhor posição alcançada por alguma
    descrição da classe. 'nomeado' quando a posição é de 1 a k."""
    termos = classes_clap.get(classe)
    if not termos:
        return 'sem correspondência', None, None, None
    melhor = (None, 10 ** 6, None)
    for j in janelas:
        pos = posicoes(j)
        for t in termos:
            if pos[t] < melhor[1]:
                melhor = (t, pos[t], j['inicio'])
    situacao = 'nomeado' if melhor[1] <= k else 'não nomeado'
    return situacao, melhor[0], melhor[1], melhor[2]

In [8]:
linhas = []
for e in EVENTOS:
    for classe in e['componentes']:
        for modelo in ['panns', 'clap']:
            for condicao in ['sem', 'com']:
                janelas = [j for j in EXECUCOES[(e['faixa'], modelo, condicao)] if sobrepoe(j, e['inicio'], e['fim'])]
                if modelo == 'panns':
                    situacao, rotulo, valor, t = nomeacao_panns(classe, janelas, PANNS_CLASSES)
                else:
                    situacao, rotulo, valor, t = nomeacao_clap(classe, janelas, CLAP_CLASSES)
                linhas.append({'id': e['id'], 'faixa': e['faixa'], 'classe': classe, 'modelo': modelo,
                               'condicao': condicao, 'situacao': situacao, 'rotulo': rotulo, 'valor': valor,
                               'tempo': mmss(t) if t is not None else None})

NOMEACOES = pd.DataFrame(linhas)
NOMEACOES.to_csv(os.path.join(PASTA_MEDIDAS, 'eventos_nomeados.csv'), index=False)
print(len(NOMEACOES), 'linhas (componente × modelo × condição)')

212 linhas (componente × modelo × condição)


### 4.1 Quadro evento a evento

Uma linha por componente de evento e uma coluna por modelo e condição. No PANNs, a célula traz o rótulo e a probabilidade. No CLAP, a descrição e a posição (#1 a #28). A marca **✓** indica nomeado, **~** indício (só no PANNs), **·** não nomeado e **—** sem correspondência.

In [9]:
MARCA = {'nomeado': '✓', 'indício': '~', 'não nomeado': '·', 'sem correspondência': '—'}

def celula(l):
    if l['situacao'] == 'sem correspondência':
        return '—'
    if pd.isna(l['rotulo']):
        return '·'
    valor = f"{l['valor']:.2f}" if l['modelo'] == 'panns' else f"#{int(l['valor'])}"
    return f"{MARCA[l['situacao']]} {l['rotulo']} {valor}"

quadro = NOMEACOES.assign(celula=NOMEACOES.apply(celula, axis=1),
                          execucao=NOMEACOES['modelo'] + ' ' + NOMEACOES['condicao'])
quadro = quadro.pivot_table(index=['faixa', 'id', 'classe'], columns='execucao', values='celula',
                            aggfunc='first', sort=False)[['panns sem', 'panns com', 'clap sem', 'clap com']]
pd.set_option('display.max_colwidth', 45)
pd.set_option('display.width', 250)
for n in range(1, 6):
    print(f'\nFaixa {n:02d}')
    print(quadro.loc[n].to_string())


Faixa 01
execucao                                                panns sem      panns com                                 clap sem                                 clap com
id   classe                                                                                                                                                       
1.1  aplauso                                      ✓ Applause 0.31              ·                            · applause #9                            · applause #9
1.2  voz alterada                                               ·              ·                   · a distorted voice #9                   ✓ a distorted voice #1
1.3  aplauso                                      ✓ Applause 0.81              ·                            · applause #4                            · applause #8
1.4  pausa                                                      —              —                                        —                                        —
1.5  aplauso

**Resultado.** O quadro mostra, evento a evento, que os dois modelos nomeiam coisas diferentes. O PANNs sem filtro nomeia sete dos oito aplausos da faixa 01 e nenhum vento, e o CLAP nomeia os cinco ventos e nenhum aplauso, com "applause" entre o 4º e o 22º lugar. As viradas de página (3.1 a 3.3), os "muito bem!" e a tosse não são nomeados por nenhum modelo, em nenhuma condição. Na faixa 04, o surdo só é nomeado pelo CLAP com filtro, por "a bass drum", e o primeiro tamborim, pela palavra "tamborim" sem descrição.

### 4.2 Proporção de eventos nomeados por classe

Para cada classe, quantos componentes foram nomeados em cada modelo e condição, sobre o total de componentes daquela classe na tabela.

In [10]:
avaliaveis = NOMEACOES[NOMEACOES['situacao'] != 'sem correspondência']
por_classe = (avaliaveis.assign(nomeado=avaliaveis['situacao'] == 'nomeado',
                                execucao=avaliaveis['modelo'] + ' ' + avaliaveis['condicao'])
              .groupby(['classe', 'execucao'])['nomeado'].agg(['sum', 'count']))
por_classe = (por_classe['sum'].astype(str) + '/' + por_classe['count'].astype(str)).unstack('execucao')
por_classe = por_classe[['panns sem', 'panns com', 'clap sem', 'clap com']].fillna('—')
por_classe.to_csv(os.path.join(PASTA_MEDIDAS, 'nomeados_por_classe.csv'))
print(por_classe.to_string())
sem = NOMEACOES[NOMEACOES['situacao'] == 'sem correspondência']
print('\nSem correspondência:', sorted(set(sem['classe'] + ' (' + sem['modelo'] + ')')))

execucao          panns sem panns com clap sem clap com
classe                                                 
aplauso                 7/8       0/8      0/8      0/8
canto                   0/1       0/1      1/1      1/1
estalos regulares       0/1       0/1      1/1      1/1
fala                    5/5       5/5      4/5      4/5
grito de plateia        0/4       0/4      0/4      0/4
música                  5/5       3/5      5/5      5/5
papel                   0/3       0/3      0/3      0/3
passos                  0/1       0/1      1/1      0/1
pulos                     —         —      2/2      2/2
ruído de disco          1/8       0/8      7/8      5/8
sino                    1/1       0/1      0/1      0/1
surdo                   0/1       0/1      0/1      1/1
tamborim                0/2       0/2      1/2      1/2
tosse                   0/1       0/1      0/1      0/1
vento                   0/5       0/5      5/5      5/5
voz alterada            0/1       0/1      0/1  

**Resultado.** A proporção por classe resume o quadro anterior. O "nomeado" do CLAP precisa ser lido com a seção 8, porque uma descrição pode ficar entre as três primeiras tanto no evento quanto fora dele, como "a distorted voice" com filtro na faixa 01 e "women singing" na faixa 05.

### 4.3 Efeito do filtro

Compara, evento a evento e em cada modelo, a situação sem e com filtro. A primeira tabela conta as combinações, e a segunda lista os componentes cuja situação muda com o filtro.

In [11]:
comp = NOMEACOES[NOMEACOES['situacao'] != 'sem correspondência'].pivot_table(
    index=['modelo', 'id', 'classe'], columns='condicao', values='situacao', aggfunc='first')
comp['nomeado sem'] = comp['sem'] == 'nomeado'
comp['nomeado com'] = comp['com'] == 'nomeado'
print(pd.crosstab([comp.index.get_level_values('modelo'), comp['nomeado sem']], comp['nomeado com'],
                  rownames=['modelo', 'nomeado sem filtro'], colnames=['nomeado com filtro']), '\n')
mudam = comp[comp['nomeado sem'] != comp['nomeado com']][['sem', 'com']]
print(mudam.to_string())

nomeado com filtro         False  True 
modelo nomeado sem filtro              
clap   False                  20      2
       True                    3     25
panns  False                  29      0
       True                   11      8 

condicao                            sem          com
modelo id   classe                                  
clap   1.2  voz alterada    não nomeado      nomeado
       2.4  ruído de disco      nomeado  não nomeado
       4.2  surdo           não nomeado      nomeado
       5.1  ruído de disco      nomeado  não nomeado
       5.9  passos              nomeado  não nomeado
panns  1.1  aplauso             nomeado  não nomeado
       1.12 aplauso             nomeado  não nomeado
       1.13 ruído de disco      nomeado  não nomeado
       1.3  aplauso             nomeado  não nomeado
       1.5  aplauso             nomeado  não nomeado
       1.7  aplauso             nomeado  não nomeado
       1.8  aplauso             nomeado  não nomeado
       1.9  apla

**Resultado.** Dos 19 componentes nomeados pelo PANNs sem filtro, 11 deixam de sê-lo com filtro, entre eles todos os aplausos e o sino, e nenhum passa a ser nomeado só com filtro. No CLAP, o efeito é misto, com dois componentes ganhos (voz alterada e surdo) e três perdidos (dois de ruído de disco e os passos).

## **5. Faixa 04 · as categorias dos modelos e a taxonomia de Almirante**

*Nos trechos em que soam o surdo (4.2) e os tamborins (4.4 e 4.6), e no trecho final com pulos (4.7), mostra os rótulos mais prováveis do PANNs e a posição de cada termo de Almirante no CLAP, conforme a seção 7 do desenho. O PANNs é lido pela probabilidade média nas janelas do trecho. O CLAP, pela posição mediana e pela melhor posição de cada termo entre as 28 descrições, comparando os termos sem descrição (tamborim, surdo, batucada, samba) com os descritos e com os distratores de percussão.*

In [12]:
TRECHOS_04 = [e for e in EVENTOS if e['faixa'] == 4 and e['classe'] in ('surdo', 'tamborim', 'pulos')]

print('PANNs · os seis rótulos de maior probabilidade média em cada trecho')
for e in TRECHOS_04:
    for condicao in ['sem', 'com']:
        janelas = [j for j in EXECUCOES[(4, 'panns', condicao)] if sobrepoe(j, e['inicio'], e['fim'])]
        media = pd.DataFrame([j['rotulos'] for j in janelas]).fillna(0).mean().sort_values(ascending=False)
        print(f"   {e['id']} {e['classe']} ({mmss(e['inicio'])} a {mmss(e['fim'])}) · {condicao}:",
              ' · '.join(f'{r} {p:.2f}' for r, p in media.head(6).items()))

PANNs · os seis rótulos de maior probabilidade média em cada trecho
   4.2 surdo (0:10,7 a 0:19,9) · sem: Speech 0.17 · Music 0.09 · Inside, small room 0.07 · Sizzle 0.05 · Frying (food) 0.04 · Musical instrument 0.03
   4.2 surdo (0:10,7 a 0:19,9) · com: Speech 0.48 · Speech synthesizer 0.14 · Heart sounds, heartbeat 0.08 · Heart murmur 0.07 · Music 0.05 · Narration, monologue 0.04
   4.4 tamborim (0:23,9 a 0:40,0) · sem: Music 0.72 · Musical instrument 0.24 · Wood block 0.14 · Speech 0.12 · Percussion 0.11 · Drum 0.06
   4.4 tamborim (0:23,9 a 0:40,0) · com: Speech 0.36 · Music 0.21 · Speech synthesizer 0.04 · Musical instrument 0.03 · Inside, small room 0.03 · Vehicle 0.03
   4.6 tamborim (0:45,0 a 0:57,0) · sem: Music 0.72 · Musical instrument 0.27 · Speech 0.10 · Guitar 0.07 · Plucked string instrument 0.07 · Synthesizer 0.06
   4.6 tamborim (0:45,0 a 0:57,0) · com: Speech 0.26 · Music 0.04 · Animal 0.04 · Biting 0.03 · Chewing, mastication 0.03 · Vehicle 0.03
   4.7 pulos (0:57,0

**Resultado.** Nenhum rótulo de tambor aparece entre os seis mais prováveis do surdo, que o PANNs aproxima de fala, música, "Sizzle" e "Frying (food)", e, com filtro, de "Heart sounds, heartbeat". Nos tamborins, o modelo fica em "Music" e "Musical instrument", com "Wood block" como nome mais próximo do primeiro tamborim. Com filtro, a fala domina todos os trechos, e nos pulos surgem "Biting" e "Chewing".

In [13]:
TERMOS_ALMIRANTE = ['tamborim', 'surdo', 'batucada', 'samba',
                    'a tamborim, a small Brazilian frame drum played with a stick',
                    'a surdo, a large Brazilian bass drum',
                    'a batucada, a Brazilian samba percussion group',
                    'a tambourine', 'a snare drum', 'a bass drum', 'music']

linhas = []
for e in TRECHOS_04:
    for condicao in ['sem', 'com']:
        janelas = [j for j in EXECUCOES[(4, 'clap', condicao)] if sobrepoe(j, e['inicio'], e['fim'])]
        pos = pd.DataFrame([posicoes(j) for j in janelas])
        for termo in TERMOS_ALMIRANTE:
            linhas.append({'trecho': f"{e['id']} {e['classe']}", 'condicao': condicao, 'termo': termo[:32],
                           'mediana': int(pos[termo].median()), 'melhor': int(pos[termo].min())})

ALMIRANTE = pd.DataFrame(linhas)
ALMIRANTE.to_csv(os.path.join(PASTA_MEDIDAS, 'faixa04_termos_clap.csv'), index=False)
tabela = ALMIRANTE.assign(pos=ALMIRANTE['mediana'].astype(str) + ' (' + ALMIRANTE['melhor'].astype(str) + ')')
print('CLAP · posição mediana (melhor posição) de cada termo, entre 28 descrições')
print(tabela.pivot_table(index='termo', columns=['trecho', 'condicao'], values='pos', aggfunc='first',
                         sort=False).to_string())

CLAP · posição mediana (melhor posição) de cada termo, entre 28 descrições
trecho                           4.2 surdo          4.4 tamborim          4.6 tamborim          4.7 pulos         
condicao                               sem      com          sem      com          sem      com       sem      com
termo                                                                                                             
tamborim                            10 (6)    8 (3)        5 (2)    1 (1)        5 (5)    8 (4)     6 (5)   10 (9)
surdo                                7 (6)    9 (3)        9 (6)    5 (2)        9 (6)    6 (5)     9 (8)    5 (4)
batucada                             5 (2)  17 (10)        5 (2)    9 (5)        7 (6)  14 (10)     7 (5)   13 (7)
samba                                9 (2)  13 (11)        3 (2)    2 (1)        8 (4)   13 (9)     7 (4)  15 (13)
a tamborim, a small Brazilian fr   28 (26)  21 (16)      25 (18)   13 (8)      26 (19)  26 (22)   26 (19)  26 (19)
a sur

**Resultado.** A palavra de Almirante sozinha fica sempre mais perto do trecho do que a mesma palavra com a descrição escrita pela pesquisadora, com uma exceção ("a surdo, a large Brazilian bass drum" no surdo com filtro). "Tamborim" tem sua melhor posição no primeiro tamborim (mediana 1 com filtro), enquanto "surdo" fica em 7º e 9º lugar no próprio trecho do surdo. A descrição em inglês do conhecimento sobre o instrumento afasta o modelo do som.

## **6. Aplausos e tosse da faixa 01**

*Cruza os eventos de aplauso, grito de plateia e tosse da tabela de referência com os rótulos dos detectores e com os segmentos em que o Whisper escreveu "Aplausos" na fase 3, conforme a seção 7 do desenho. Os JSON da fase 3 são lidos da pasta `resultados/fase3`.*

In [14]:
aplausos_whisper = []
for c in 'ABCD':
    with open(os.path.join(PASTA_FASE3, f'faixa01_{c}.json'), encoding='utf-8') as f:
        for s in json.load(f)['segmentos']:
            if 'aplau' in s['text'].lower():
                aplausos_whisper.append({'cond.': c, 'inicio': s['start'], 'fim': s['end'], 'texto': s['text'].strip()})

linhas = []
for e in [e for e in EVENTOS if e['faixa'] == 1 and {'aplauso', 'grito de plateia', 'tosse'} & set(e['componentes'])]:
    linha = {'id': e['id'], 'classe': e['classe'], 'tempo': f"{mmss(e['inicio'])}–{mmss(e['fim'])}"}
    for condicao in ['sem', 'com']:
        janelas = [j for j in EXECUCOES[(1, 'panns', condicao)] if sobrepoe(j, e['inicio'], e['fim'])]
        linha[f'Applause {condicao}'] = round(max(j['rotulos'].get('Applause', 0) for j in janelas), 2)
        janelas = [j for j in EXECUCOES[(1, 'clap', condicao)] if sobrepoe(j, e['inicio'], e['fim'])]
        linha[f'applause CLAP {condicao}'] = '#' + str(min(posicoes(j)['applause'] for j in janelas))
    linha['Whisper'] = ', '.join(sorted({a['cond.'] for a in aplausos_whisper
                                          if a['inicio'] < e['fim'] and a['fim'] > e['inicio']})) or '—'
    linhas.append(linha)
print(pd.DataFrame(linhas).to_string(index=False), '\n')

print('Segmentos do Whisper com "Aplausos" (fase 3):')
for a in aplausos_whisper:
    dentro = [e['id'] for e in EVENTOS if e['faixa'] == 1 and 'aplauso' in e['componentes']
              and a['inicio'] < e['fim'] and a['fim'] > e['inicio']]
    print(f"   {a['cond.']} · {mmss(a['inicio'])}–{mmss(a['fim'])} · {a['texto']} · evento: {', '.join(dentro) or 'nenhum'}")

  id                     classe         tempo  Applause sem applause CLAP sem  Applause com applause CLAP com Whisper
 1.1                    aplauso 0:36,6–0:46,1          0.31                #9             0                #9       —
 1.3                    aplauso 1:34,6–1:41,0          0.81                #4             0                #8       —
 1.5 aplauso e grito de plateia 2:18,2–2:26,7          0.78                #6             0                #9       —
 1.7 aplauso e grito de plateia 3:17,2–3:25,9          0.46                #6             0               #10       —
 1.8 aplauso e grito de plateia 3:55,1–4:03,8          0.82                #7             0                #9       —
 1.9                    aplauso 4:11,0–4:18,0          0.66                #7             0               #12       A
1.10                    aplauso 4:51,3–4:56,9          0.10               #16             0               #22       A
1.11                      tosse 4:55,0–4:56,0          0

**Resultado.** O PANNs aponta aplauso em sete dos oito eventos sem filtro e em nenhum com filtro, e o CLAP nunca põe "applause" entre as três primeiras. O Whisper escreveu "Aplausos." apenas nos três últimos aplausos e só na condição A, e no 1.10, em que o PANNs quase não vê aplauso (0,10), a palavra foi escrita.

### 6.1 O que os detectores apontam na tosse

Mostra, janela a janela, de 4:52 a 4:59, os quatro rótulos mais altos do PANNs e as três primeiras descrições do CLAP, com e sem filtro e sem nenhuma restrição de classe. A tosse (1.11) foi marcada de 4:55,0 a 4:56,0, dentro do aplauso 1.10 (4:51,3 a 4:56,9). As janelas vizinhas entram porque a janela de 2 segundos e a marcação da escuta não coincidem necessariamente.

In [15]:
for condicao in ['sem', 'com']:
    print(f'Condição {condicao} filtro')
    for jp, jc in zip(EXECUCOES[(1, 'panns', condicao)], EXECUCOES[(1, 'clap', condicao)]):
        if 292 <= jp['inicio'] <= 297:
            panns = ' · '.join(f'{r} {p:.2f}' for r, p in list(jp['rotulos'].items())[:4])
            clap = ' · '.join(sorted(jc['semelhancas'], key=jc['semelhancas'].get, reverse=True)[:3])
            print(f"   {mmss(jp['inicio'])}–{mmss(jp['fim'])} · PANNs: {panns}")
            print(f"   {' ' * 13} · CLAP: {clap}")

Condição sem filtro
   4:52,0–4:54,0 · PANNs: Radio 0.12 · Speech 0.07 · Crackle 0.06 · Applause 0.05
                 · CLAP: the surface noise of an old record · clicks of a scratched record · a skipping record
   4:53,0–4:55,0 · PANNs: Applause 0.10 · Radio 0.07 · Speech 0.05 · Whip 0.03
                 · CLAP: the surface noise of an old record · clicks of a scratched record · a skipping record
   4:54,0–4:56,0 · PANNs: Animal 0.13 · Speech 0.08 · Crow 0.08 · Applause 0.06
                 · CLAP: the surface noise of an old record · clicks of a scratched record · a skipping record
   4:55,0–4:57,0 · PANNs: Speech 0.13 · Animal 0.11 · Crow 0.09 · Caw 0.07
                 · CLAP: the surface noise of an old record · clicks of a scratched record · a skipping record
   4:56,0–4:58,0 · PANNs: Sheep 0.57 · Bleat 0.41 · Speech 0.29 · Animal 0.22
                 · CLAP: the surface noise of an old record · a skipping record · clicks of a scratched record
   4:57,0–4:59,0 · PANNs: Speec

**Resultado.** A tosse (4:55 a 4:56) aparece no PANNs como vocalização animal, primeiro "Crow" e "Caw", com valores baixos, e depois "Sheep" (0,57) e "Bleat" (0,41) na janela de 4:56 a 4:58, que contém a segunda parte da tosse, conforme a escuta dirigida (trecho 5). "Cough" não aparece em nenhuma janela. Com filtro, os animais dão lugar a "Clicking", "Chewing" e "Whispering".

## **7. Rótulos registrados à parte**

*Acompanha os rótulos da seção 6 do desenho que não correspondem a nenhuma classe. Para o PANNs, mostra a probabilidade máxima de cada rótulo em cada faixa e condição, quando passa de 0,1. Para o CLAP, conta em quantas janelas cada descrição ficou em primeiro lugar e entre as três primeiras. Em seguida, confere os quatro casos que motivaram o registro, nos eventos a que se referem.*

In [16]:
print('PANNs · probabilidade máxima (tempo), quando passa de 0,1')
for rotulo, motivo in A_PARTE_PANNS.items():
    for n in range(1, 6):
        for condicao in ['sem', 'com']:
            janelas = EXECUCOES[(n, 'panns', condicao)]
            p, t = max((j['rotulos'].get(rotulo, 0), j['inicio']) for j in janelas)
            if p >= 0.1:
                print(f'   {rotulo} ({motivo}) · faixa {n:02d} {condicao} · {p:.2f} em {mmss(t)}')

print('\nCLAP · janelas em primeiro lugar / entre as três primeiras')
for termo, motivo in A_PARTE_CLAP.items():
    contagem = []
    for n in range(1, 6):
        for condicao in ['sem', 'com']:
            pos = [posicoes(j)[termo] for j in EXECUCOES[(n, 'clap', condicao)]]
            contagem.append(f'{n:02d} {condicao} {sum(p == 1 for p in pos)}/{sum(p <= 3 for p in pos)}')
    print(f'   {termo} ({motivo}) ·', ' · '.join(contagem))

PANNs · probabilidade máxima (tempo), quando passa de 0,1
   Silence (pausa) · faixa 01 sem · 0.26 em 5:33,5
   Silence (pausa) · faixa 01 com · 0.26 em 5:33,5
   Silence (pausa) · faixa 03 sem · 0.26 em 6:00,1
   Silence (pausa) · faixa 03 com · 0.26 em 6:00,1
   Silence (pausa) · faixa 05 com · 0.26 em 5:01,0
   Door (passos (alternativa)) · faixa 04 com · 0.13 em 0:50,0

CLAP · janelas em primeiro lugar / entre as três primeiras
   a tambourine (tamborim (homônimo)) · 01 sem 0/0 · 01 com 0/0 · 02 sem 0/0 · 02 com 0/0 · 03 sem 0/0 · 03 com 1/2 · 04 sem 0/0 · 04 com 0/0 · 05 sem 0/0 · 05 com 0/0
   silence (pausa) · 01 sem 1/1 · 01 com 1/2 · 02 sem 0/0 · 02 com 0/0 · 03 sem 1/2 · 03 com 1/8 · 04 sem 0/0 · 04 com 0/0 · 05 sem 9/39 · 05 com 169/325
   a door knocking (passos (alternativa)) · 01 sem 0/0 · 01 com 0/0 · 02 sem 0/0 · 02 com 0/0 · 03 sem 0/0 · 03 com 0/0 · 04 sem 0/0 · 04 com 0/6 · 05 sem 0/0 · 05 com 2/16
   ocean waves (surpresa da fase 1) · 01 sem 0/0 · 01 com 2/22 · 02 s

In [17]:
CASOS = [('Tambourine', 'a tambourine', ['4.4', '4.6']),
         ('Silence', 'silence', ['1.4', '1.6']),
         ('Door', 'a door knocking', ['5.9']), ('Knock', 'a door knocking', ['5.9']),
         ('Ocean', 'ocean waves', ['5.11']), ('Waves, surf', 'ocean waves', ['5.11'])]

for rotulo, termo, ids in CASOS:
    for i in ids:
        e = next(e for e in EVENTOS if e['id'] == i)
        partes = []
        for condicao in ['sem', 'com']:
            janelas = [j for j in EXECUCOES[(e['faixa'], 'panns', condicao)] if sobrepoe(j, e['inicio'], e['fim'])]
            partes.append(f"{rotulo} {condicao} {max(j['rotulos'].get(rotulo, 0) for j in janelas):.2f}")
            janelas = [j for j in EXECUCOES[(e['faixa'], 'clap', condicao)] if sobrepoe(j, e['inicio'], e['fim'])]
            partes.append(f"{termo} {condicao} #{min(posicoes(j)[termo] for j in janelas)}")
        print(f"{i} {e['classe']} ·", ' · '.join(partes))

4.4 tamborim · Tambourine sem 0.00 · a tambourine sem #10 · Tambourine com 0.00 · a tambourine com #7
4.6 tamborim · Tambourine sem 0.00 · a tambourine sem #12 · Tambourine com 0.00 · a tambourine com #13
1.4 pausa · Silence sem 0.00 · silence sem #15 · Silence com 0.11 · silence com #9
1.6 pausa · Silence sem 0.00 · silence sem #14 · Silence com 0.09 · silence com #12
5.9 passos · Door sem 0.03 · a door knocking sem #5 · Door com 0.00 · a door knocking com #8
5.9 passos · Knock sem 0.02 · a door knocking sem #5 · Knock com 0.00 · a door knocking com #8
5.11 música · Ocean sem 0.00 · ocean waves sem #6 · Ocean com 0.00 · ocean waves com #2
5.11 música · Waves, surf sem 0.00 · ocean waves sem #6 · Waves, surf com 0.00 · ocean waves com #2


**Resultado.** "Tambourine" não aparece nos tamborins. "Silence" só se aproxima de 0,3 na última janela das faixas 01 e 03, onde a escuta (trechos 6 e 20) encontrou silêncio completo depois do fim do som do disco. Os dois modelos preferem os passos à batida na porta no evento 5.9. "Ocean" e "Waves, surf" ficam em zero no 5.11, e, sem filtro, "ocean waves" só entra entre as três primeiras do CLAP nas passagens de vento da faixa 05.

## **8. O excesso de apontamentos do CLAP**

*O CLAP não diz se um som está presente, apenas ordena as descrições do vocabulário pela semelhança com o trecho. Por isso, toda janela tem três descrições apontadas, haja ou não algo que corresponda a elas. A tabela mostra, para cada faixa e condição, as descrições que mais vezes ficaram em primeiro lugar e entre as três primeiras, e, para as que correspondem a uma classe, quantas dessas janelas ficam fora de qualquer evento da classe na tabela de referência. É a medida que motivou a decisão de 20/10/2026 sobre a escuta dirigida (seção 10).*

In [18]:
linhas = []
for n in range(1, 6):
    for condicao in ['sem', 'com']:
        janelas = EXECUCOES[(n, 'clap', condicao)]
        for termo in janelas[0]['semelhancas']:
            pos = [posicoes(j)[termo] for j in janelas]
            classes = INVERSO_CLAP.get(termo, set())
            fora = [j for j, p in zip(janelas, pos) if p <= 3 and classes and not any(
                e['faixa'] == n and c in e['componentes'] and sobrepoe(j, e['inicio'], e['fim'])
                for e in EVENTOS for c in classes)]
            linhas.append({'faixa': n, 'condicao': condicao, 'termo': termo, 'janelas': len(janelas),
                           '1º lugar': sum(p == 1 for p in pos), 'top 3': sum(p <= 3 for p in pos),
                           'top 3 fora da classe': len(fora) if classes else None})

FREQ_CLAP = pd.DataFrame(linhas).astype({'top 3 fora da classe': 'Int64'})
FREQ_CLAP.to_csv(os.path.join(PASTA_MEDIDAS, 'clap_frequencias.csv'), index=False)
for (n, condicao), g in FREQ_CLAP.groupby(['faixa', 'condicao'], sort=False):
    g = g.sort_values('top 3', ascending=False).head(5)
    print(f"\nFaixa {n:02d} {condicao} · {g['janelas'].iloc[0]} janelas")
    print(g[['termo', '1º lugar', 'top 3', 'top 3 fora da classe']].to_string(index=False))


Faixa 01 sem · 335 janelas
                             termo  1º lugar  top 3  top 3 fora da classe
the surface noise of an old record       241    333                     0
             a man giving a speech        88    228                   228
                 a skipping record         2    151                   151
      clicks of a scratched record         0    108                   108
                          batucada         2    106                  <NA>

Faixa 01 com · 335 janelas
                termo  1º lugar  top 3  top 3 fora da classe
                surdo       154    305                   305
    a skipping record        64    208                   208
    a distorted voice        64    191                   182
a man giving a speech        40    114                   114
                music         2     51                    51

Faixa 02 sem · 201 janelas
                             termo  1º lugar  top 3  top 3 fora da classe
the surface noise of an old reco

**Resultado.** Sem filtro, a descrição do ruído do disco fica em primeiro lugar em quase todas as janelas das faixas 01, 02 e 04. Com filtro, a palavra "surdo" ocupa esse lugar na faixa 01 (154 janelas) e na leitura do policial (207 de 362), e a faixa 05 passa a ser descrita como "silence" e "ocean waves". O CLAP sempre escolhe alguma descrição, e a seleção por posição, registrada no desenho, apontaria alguma coisa em toda janela.

## **9. Confronto com as expectativas da fase 1**

*Confere, um a um, os rótulos que a pesquisadora esperava ver aparecer (`docs/fase1/05_expectativas.md`), com a probabilidade máxima do PANNs no intervalo indicado e o tempo da janela em que ocorreu. Em seguida, recolhe da seção 4 a situação dos fenômenos que ela apostou que nenhum modelo nomearia e o do critério de surpresa ("mar" entre 4:38 e 5:03 da faixa 05).*

In [19]:
ESPERADOS = [
    (1, ['Applause'], 'aplausos (todos os eventos de aplauso)', None),
    (4, ['Drum', 'Percussion', 'Snare drum', 'Bass drum', 'Tambourine'], 'surdo e tamborins', ('0:10,7', '0:57')),
    (5, ['Hiss'], 'início da gravação', ('0:00', '0:03')),
    (5, ['Christian music'], 'música religiosa', ('0:04', '1:03')),
    (5, ['Wind'], 'vento 5.3', ('1:04', '1:25')),
    (5, ['Music of Latin America'], 'música tradicional', ('1:27', '1:51')),
    (5, ['Wind'], 'vento 5.7', ('3:19', '3:50')),
    (5, ['Church bell'], 'sino', ('3:52', '4:08')),
    (5, ['Walk, footsteps'], 'passos', ('4:20', '4:27')),
    (5, ['Wind'], 'vento 5.10', ('4:28', '4:37')),
    (5, ['Exciting music'], 'música de suspense', ('4:38', '5:03')),
    (5, ['Ocean', 'Waves, surf'], 'surpresa ("mar")', ('4:38', '5:03')),
]

for n, rotulos, nome, intervalo in ESPERADOS:
    if intervalo is None:
        intervalos = [(e['inicio'], e['fim']) for e in EVENTOS if e['faixa'] == n and 'aplauso' in e['componentes']]
    else:
        intervalos = [(segundos(intervalo[0]), segundos(intervalo[1]))]
    for rotulo in rotulos:
        partes = []
        for condicao in ['sem', 'com']:
            janelas = [j for j in EXECUCOES[(n, 'panns', condicao)] if any(sobrepoe(j, a, b) for a, b in intervalos)]
            p, t = max((j['rotulos'].get(rotulo, 0), j['inicio']) for j in janelas)
            partes.append(f'{condicao} {p:.2f} em {mmss(t)}' if p >= 0.01 else f'{condicao} < 0.01')
        print(f'Faixa {n:02d} · {nome} · {rotulo} ·', ' · '.join(partes))

Faixa 01 · aplausos (todos os eventos de aplauso) · Applause · sem 0.82 em 3:57,0 · com < 0.01
Faixa 04 · surdo e tamborins · Drum · sem 0.15 em 0:31,0 · com 0.02 em 0:26,0
Faixa 04 · surdo e tamborins · Percussion · sem 0.26 em 0:31,0 · com 0.02 em 0:26,0
Faixa 04 · surdo e tamborins · Snare drum · sem < 0.01 · com < 0.01
Faixa 04 · surdo e tamborins · Bass drum · sem < 0.01 · com < 0.01
Faixa 04 · surdo e tamborins · Tambourine · sem < 0.01 · com < 0.01
Faixa 05 · início da gravação · Hiss · sem < 0.01 · com < 0.01
Faixa 05 · música religiosa · Christian music · sem < 0.01 · com < 0.01
Faixa 05 · vento 5.3 · Wind · sem 0.01 em 1:15,0 · com < 0.01
Faixa 05 · música tradicional · Music of Latin America · sem < 0.01 · com < 0.01
Faixa 05 · vento 5.7 · Wind · sem < 0.01 · com < 0.01
Faixa 05 · sino · Church bell · sem 0.03 em 3:57,0 · com < 0.01
Faixa 05 · passos · Walk, footsteps · sem < 0.01 · com < 0.01
Faixa 05 · vento 5.10 · Wind · sem 0.02 em 4:27,0 · com < 0.01
Faixa 05 · música d

In [20]:
APOSTAS = {'virada de página': ['3.1', '3.2', '3.3'], 'mulheres cantando': ['5.6'],
           '"muito bem!" (grito de plateia)': ['1.5', '1.7', '1.8', '1.12']}
for nome, ids in APOSTAS.items():
    g = NOMEACOES[NOMEACOES['id'].isin(ids) & (NOMEACOES['classe'] != 'aplauso')]
    print(f'\n{nome}')
    print(g.assign(celula=g.apply(celula, axis=1)).pivot_table(
        index='id', columns=['modelo', 'condicao'], values='celula', aggfunc='first').to_string())


virada de página
modelo                                  clap                                     panns               
condicao                                 com                                 sem   com            sem
id                                                                                                   
3.1       · a page of paper being turned #25  · a page of paper being turned #21     ·  · Rustle 0.01
3.2       · a page of paper being turned #20  · a page of paper being turned #13     ·  · Rustle 0.03
3.3       · a page of paper being turned #20  · a page of paper being turned #16     ·              ·

mulheres cantando
modelo                  clap                               panns                       
condicao                 com                 sem             com                    sem
id                                                                                     
5.6       ✓ women singing #1  ✓ women singing #1  · Singing 0.01  ~ Female singing 0.23

**Resultado.** Dos rótulos que a pesquisadora esperava ver, só "Applause" aparece, e só sem filtro. Os rótulos de gênero musical, de sino, de passos, de vento e de percussão específica ficam abaixo de 0,03. A virada de página e o "muito bem!" não são nomeados, como ela apostou, e as mulheres cantando aparecem em primeiro lugar no CLAP, com a ressalva de especificidade da seção 8.

## **10. Lista para a escuta dirigida**

*Aplica a seção 9 do desenho experimental, com a alteração decidida em 20/10/2026 e registrada no diário da fase. No PANNs, os critérios ficam como foram registrados. Entra o rótulo correspondente a uma classe, apontado (0,3 ou mais) numa janela que não se sobrepõe a nenhum evento dessa classe (critério 1), e o rótulo sem classe com 0,5 ou mais (critério 2). A fala nas faixas 01 a 03 e o ruído de disco em todas as faixas ficam fora do critério 1. Quando um rótulo corresponde a mais de uma classe (Drum, por exemplo, ao surdo e ao tamborim), ele só entra se nenhuma delas estiver presente na janela. No CLAP, só entram as janelas em que a mesma descrição fica em primeiro lugar com e sem filtro, e que atendem ao critério 1 ou ao critério 2. Por fim, itens da mesma faixa separados por até 5 segundos formam um só trecho.*

Motivo da alteração. Com a regra registrada para o CLAP, que conta como apontadas as três descrições mais semelhantes de cada janela, a lista chegaria a 4.831 itens, cobrindo cerca de 23 minutos, praticamente o corpus inteiro (seção 8). O excesso de apontamentos do CLAP passa a ser tratado como resultado, e não como lista de escuta.

In [21]:
def explicado(n, classe, janela, eventos):
    """A classe está presente na janela segundo a referência (ou fica fora do critério 1)."""
    if n in SEMPRE_PRESENTES.get(classe, set()):
        return True
    return any(e['faixa'] == n and classe in e['componentes'] and sobrepoe(janela, e['inicio'], e['fim'])
               for e in eventos)


def candidatos_panns(n, condicao, janelas, inverso, eventos):
    """Itens da escuta dirigida vindos do PANNs (seção 9 do desenho)."""
    itens = []
    for j in janelas:
        for r, p in j['rotulos'].items():
            if p < LIMIAR_APONTADO:
                continue
            if r in inverso:
                if not all(explicado(n, c, j, eventos) for c in inverso[r]):
                    itens.append(dict(faixa=n, inicio=j['inicio'], fim=j['fim'], modelo='PANNs',
                                      condicao=condicao, criterio=1, rotulo=r, valor=round(p, 2)))
            elif p >= LIMIAR_SEM_CLASSE:
                itens.append(dict(faixa=n, inicio=j['inicio'], fim=j['fim'], modelo='PANNs',
                                  condicao=condicao, criterio=2, rotulo=r, valor=round(p, 2)))
    return itens


def candidatos_clap(n, janelas_sem, janelas_com, inverso, eventos):
    """Itens da escuta dirigida vindos do CLAP (decisão de 20/10/2026). Só entram as janelas em
    que a mesma descrição fica em primeiro lugar com e sem filtro."""
    itens = []
    for a, b in zip(janelas_sem, janelas_com):
        t = primeiro(a)
        if t != primeiro(b):
            continue
        if t in inverso:
            if all(explicado(n, c, a, eventos) for c in inverso[t]):
                continue
            criterio = 1
        else:
            criterio = 2
        itens.append(dict(faixa=n, inicio=a['inicio'], fim=a['fim'], modelo='CLAP',
                          condicao='sem e com', criterio=criterio, rotulo=t, valor=1))
    return itens


def agrupar(itens, folga=FOLGA_AGRUPAMENTO):
    """Numera os trechos da escuta. Itens da mesma faixa que se sobrepõem ou que estão a até
    'folga' segundos um do outro formam um só trecho. Devolve os itens com a chave 'trecho'."""
    ordenados = sorted(itens, key=lambda x: (x['faixa'], x['inicio'], x['fim']))
    saida, trecho, faixa, fim = [], 0, None, None
    for x in ordenados:
        if x['faixa'] != faixa or x['inicio'] > fim + folga:
            trecho += 1
            faixa, fim = x['faixa'], x['fim']
        fim = max(fim, x['fim'])
        saida.append({**x, 'trecho': trecho})
    return saida

In [22]:
itens = []
for n in range(1, 6):
    for condicao in ['sem', 'com']:
        itens += candidatos_panns(n, condicao, EXECUCOES[(n, 'panns', condicao)], INVERSO_PANNS, EVENTOS)
    itens += candidatos_clap(n, EXECUCOES[(n, 'clap', 'sem')], EXECUCOES[(n, 'clap', 'com')], INVERSO_CLAP, EVENTOS)

ESCUTA = pd.DataFrame(agrupar(itens))
ESCUTA = ESCUTA[['trecho', 'faixa', 'inicio', 'fim', 'modelo', 'condicao', 'criterio', 'rotulo', 'valor']]
ESCUTA.to_csv(os.path.join(PASTA_MEDIDAS, 'escuta_dirigida.csv'), index=False)

trechos = ESCUTA.groupby('trecho').agg(faixa=('faixa', 'first'), inicio=('inicio', 'min'), fim=('fim', 'max'))
print(len(ESCUTA), 'itens ·', len(trechos), 'trechos ·', f"{(trechos['fim'] - trechos['inicio']).sum() / 60:.1f} minutos")
print(trechos.groupby('faixa').size().to_dict())

401 itens · 34 trechos · 6.4 minutos
{1: 6, 2: 7, 3: 7, 4: 2, 5: 12}


In [23]:
for t, g in ESCUTA.groupby('trecho'):
    print(f"\nTrecho {t} · faixa {g['faixa'].iloc[0]:02d} · {mmss(g['inicio'].min())} a {mmss(g['fim'].max())}")
    resumo = g.groupby(['modelo', 'condicao', 'criterio', 'rotulo'])['valor'].agg(['max', 'size']).reset_index()
    for _, l in resumo.iterrows():
        valor = f"{l['max']:.2f}" if l['modelo'] == 'PANNs' else '1º lugar'
        print(f"   {l['modelo']} {l['condicao']} · critério {l['criterio']} · {l['rotulo']} · {valor} · {l['size']} janela(s)")


Trecho 1 · faixa 01 · 0:00,0 a 0:06,0
   PANNs sem · critério 1 · Applause · 0.64 · 5 janela(s)

Trecho 2 · faixa 01 · 1:34,0 a 1:36,0
   PANNs com · critério 1 · Clicking · 0.35 · 1 janela(s)

Trecho 3 · faixa 01 · 4:11,0 a 4:13,0
   PANNs com · critério 1 · Clicking · 0.44 · 1 janela(s)

Trecho 4 · faixa 01 · 4:30,0 a 4:32,0
   CLAP sem e com · critério 1 · a skipping record · 1º lugar · 1 janela(s)

Trecho 5 · faixa 01 · 4:54,0 a 4:58,0
   PANNs com · critério 1 · Clicking · 0.33 · 1 janela(s)
   PANNs sem · critério 2 · Sheep · 0.57 · 1 janela(s)

Trecho 6 · faixa 01 · 5:27,0 a 5:35,5
   CLAP sem e com · critério 2 · silence · 1º lugar · 1 janela(s)
   PANNs com · critério 2 · Crunch · 0.61 · 3 janela(s)

Trecho 7 · faixa 02 · 0:02,0 a 0:19,0
   PANNs com · critério 2 · Speech synthesizer · 0.71 · 6 janela(s)

Trecho 8 · faixa 02 · 0:29,0 a 1:13,0
   PANNs com · critério 2 · Speech synthesizer · 0.67 · 14 janela(s)
   PANNs sem · critério 1 · Music · 0.64 · 4 janela(s)

Trecho 9 ·

**Resultado.** Os 34 trechos foram escutados pela pesquisadora em 21 e 22/10/2026 (`docs/fase4/03_escuta_dirigida.md`). Dos 57 rótulos ou grupos de rótulos classificados, 21 não correspondem a nada que a escuta reconheça (categoria 4), 20 nomeiam com outro nome um som presente (categoria 3) e 16 correspondem a alterações produzidas pela filtragem (categoria 5). Em dois casos, a máquina apontou algo presente e não registrado, os aplausos do início da faixa 01, único caso de surpresa pelo critério da fase 1, e o silêncio do fim do arquivo da faixa 03.

## **11. Ambiente de execução**

In [24]:
import sys, importlib.metadata as md
from datetime import datetime
print(datetime.now().strftime('%d/%m/%Y %H:%M'))
print('Python', sys.version.split()[0])
for p in ['pandas', 'panns-inference']:
    print(p, md.version(p))

23/10/2026 13:57
Python 3.13.16
pandas 2.2.3
panns-inference 0.1.1


## **12. Testes automáticos**

*Executa os testes de `tests/`, os da fase 3 e os de `tests/test_eventos.py`, que conferem, com casos de resposta conhecida, a leitura da tabela de referência, os limiares de nomeação e os critérios da escuta dirigida. As funções testadas estão em `src/eventos.py`, cópia idêntica das definidas neste caderno. A biblioteca jiwer é instalada antes, porque os testes da fase 3 dependem dela.*

In [25]:
!pip install -q jiwer==4.0.0
!cd /content/drive/MyDrive/escuta-cerrada-poc && python -m pytest -q tests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 71.5 MB/s eta 0:00:00
..................                                                       [100%]
18 passed in 1.45s


**Resultado.** Os 18 testes passaram (23/10/2026).